# scRNA-seq analysis — MHC II expression in LUAD epithelial cells

**Goal:** Characterize MHC II expression in malignant vs normal adjacent epithelial
cells from LUAD patients, and compare expression across disease contexts including
primary tumor vs metastasis, early vs late stage, and AT2 vs Club cell identity.

**Input:** Salcher et al. 2022 pan-cancer scRNA-seq atlas (`local.h5ad`), subset to
lung adenocarcinoma epithelial lineage cells. Patient-level MHC II classifications
from IHC are used in figure 2e.

**Output:** Publication figures (main + supplemental) for figure 2 and associated
supplemental panels. Processed AnnData object with Harmony-corrected UMAP saved
to `outputs/processed/` for reuse across sessions.

## Setup

In [ ]:
# standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# single-cell analysis
import anndata as ad
import scanpy as sc

# clustering and statistics
from sklearn.cluster import AgglomerativeClustering
from scipy.stats import zscore, wilcoxon, mannwhitneyu
from scipy.cluster.hierarchy import dendrogram, linkage
import scipy.sparse as sp

# utilities
from tqdm import tqdm
from pathlib import Path
import yaml

# ceiba modules
from ceiba.plot_utils import sig_label

# figure settings
sns.set(font_scale=1.8)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42  # embed fonts in PDF output — required by most journals
plt.rcParams['ps.fonttype'] = 42
plt.rcParams.update({'axes.titlesize': 20})

# paper-wide color palettes — consistent across all analyses
# MHC II pos = orange (#FF8811), MHC II neg = purple (#462255)
cmap_low_high = ['#462255FF', '#FF8811FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']  # neg first
cmap_high_low = ['#FF8811FF', '#462255FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']  # pos first
cmap_umap     = ['#9DD9D2FF', '#462255FF', '#FF8811FF', '#046E8FFF', '#D44D5CFF']  # used in UMAPs

## Data loading

In [ ]:
import yaml
from pathlib import Path

# load paths configuration
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])

# data paths
atlas_path     = cfg['datasets']['salcher_atlas']['raw']
processed_path = repo_root / cfg['outputs']['processed'] / 'luad_epithelial_harmony.h5ad'

# output paths — resolved against repo root
fig_dir   = repo_root / cfg['outputs']['figures']
table_dir = repo_root / cfg['outputs']['tables']

# create output directories if they don't exist
fig_out   = fig_dir / 'figure2'
table_out = table_dir / 'figure2'
fig_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)

# load data — uses pre-processed Harmony object if available
# full atlas load is only required on first run or if reprocessing from scratch
if processed_path.exists():
    epithelial = sc.read_h5ad(processed_path)
    print(f'Loaded pre-processed epithelial object: {epithelial.n_obs:,} cells')
else:
    adata      = sc.read_h5ad(atlas_path)
    print(f'Loaded full atlas: {adata.n_obs:,} cells')
    # create epithelial subset so downstream cells always have epithelial defined
    # Harmony reintegration will overwrite this with the corrected embedding
    epithelial = adata[
        (adata.obs['disease'] == 'lung adenocarcinoma') &
        (adata.obs['ann_coarse'] == 'Epithelial cell')
    ].copy()
    print(f'Epithelial subset created: {epithelial.n_obs:,} cells')
    print('Run the Harmony reintegration section to compute embedding and save processed object')

In [ ]:
# add cancer_noncancer label if not already present
# this column is not saved with the processed object
if 'cancer_noncancer' not in epithelial.obs.columns:
    epithelial.obs['cancer_noncancer'] = np.where(
        epithelial.obs['ann_fine'] == 'Cancer cells',
        'Cancer cells',
        'Epithelial cells',
    )
    print('Added cancer_noncancer label')

# add numeric sample ID for batch visualization if not present
if 'sample_id' not in epithelial.obs.columns:
    epithelial.obs['sample_id'] = (
        epithelial.obs['sample']
        .map({s: i for i, s in enumerate(epithelial.obs['sample'].unique())})
        .astype(float)
    )
    print('Added sample_id')

## Gene lists

In [ ]:
# core MHC II antigen presentation genes
mhc2_genes = ['HLA-DRA', 'HLA-DRB1', 'HLA-DPA1', 'HLA-DPB1',
               'HLA-DQA1', 'HLA-DQB1', 'CD74', 'CIITA']

# extended set — adds accessory/co-stimulatory molecules and S100P
# S100P is included here as a comparator marker, not an MHC II gene
extended_genes = mhc2_genes + ['CD80', 'CD86', 'ICAM1', 'HLA-DMA', 'HLA-DMB', 'S100P']

def resolve_ensembl(adata, gene_list):
    """
    Map gene symbols to Ensembl IDs via var['feature_name'].

    Parameters
    ----------
    adata : AnnData
    gene_list : list of str
        Gene symbols to resolve.

    Returns
    -------
    dict : {gene_symbol: [ensembl_id, ...]}
    """
    out = {}
    for gene in gene_list:
        hits = list(adata.var.index[adata.var['feature_name'] == gene])
        if not hits:
            print(f'Warning: {gene} not found in var["feature_name"]')
        out[gene] = hits
    return out

    
# resolve gene symbols to Ensembl IDs — requires epithelial to be loaded
mhc2_dict     = resolve_ensembl(epithelial, mhc2_genes)
extended_dict = resolve_ensembl(epithelial, extended_genes)

# flat Ensembl ID lists for expression indexing
ens_mhc2     = [item for sublist in mhc2_dict.values()    for item in sublist]
ens_extended = [item for sublist in extended_dict.values() for item in sublist]

print(f'MHC II genes resolved:   {len(ens_mhc2)} Ensembl IDs')
print(f'Extended genes resolved: {len(ens_extended)} Ensembl IDs')

## Helper functions

Plotting functions used across figure 2 panels. `plot_genes_paired_luad` and
`plot_genes_paired_luad_percent_detected` operate on the full atlas object and
perform subsetting internally. `plot_scrna_group_comparison` operates on
pre-filtered AnnData subsets and is reused for figures 2c, 2d, and supplemental
comparisons. All three are pending migration to `ceiba.plot_utils`.

In [ ]:
from ceiba.plot_utils import (
    plot_genes_paired_luad,
    plot_genes_paired_luad_percent_detected,
    plot_genes_pct_expressing_luad,
    plot_scrna_group_comparison,
    plot_celltype_comparison_luad,
)
from ceiba.stats_utils import (
    ciita_expr_by_s100p_strata_per_sample,
    ciita_expr_cell_level_tests,
)

## Epithelial lineage subsetting and Harmony reintegration

The Salcher atlas UMAP embeds all cell types together. When subsetting to
epithelial lineage cells only, the original embedding collapses into a dense
cluster that obscures within-lineage structure. A new UMAP is computed on the
epithelial subset using Harmony batch correction across studies.

This section only needs to run once. The processed object is saved to
`outputs/processed/epithelial_harmony.h5ad` and loaded automatically on
subsequent sessions.

In [ ]:
# guard — skip if processed object was already loaded
if 'epithelial' not in dir():

    # subset to LUAD epithelial lineage cells
    adenodata  = adata[adata.obs['disease'] == 'lung adenocarcinoma'].copy()
    epithelial = adenodata[adenodata.obs['ann_coarse'] == 'Epithelial cell'].copy()

    # label cancer vs non-cancer epithelial cells
    epithelial.obs['cancer_noncancer'] = np.where(
        epithelial.obs['ann_fine'] == 'Cancer cells',
        'Cancer cells',
        'Epithelial cells',
    )

    print(f'Epithelial lineage cells: {epithelial.n_obs:,}')
    print(epithelial.obs['cancer_noncancer'].value_counts())

In [ ]:
if 'X_pca_harmony' not in epithelial.obsm:
    import time
    import harmonypy

    # HVG selection
    print('Step 1/5 — highly variable genes...')
    t0 = time.time()
    sc.pp.highly_variable_genes(epithelial, n_top_genes=2000, subset=False)
    print(f'  done ({time.time()-t0:.1f}s) — {epithelial.var.highly_variable.sum()} HVGs selected')

    # PCA
    print('Step 2/5 — PCA...')
    t0 = time.time()
    sc.tl.pca(epithelial, mask_var='highly_variable')
    print(f'  done ({time.time()-t0:.1f}s)')

    # Harmony — called directly due to anndata/harmonypy version incompatibility
    # in this version of harmonypy, Z_corr is already (n_cells, n_pcs) — no transpose needed
    print('Step 3/5 — Harmony batch correction (this is the slow step)...')
    t0 = time.time()
    ho = harmonypy.run_harmony(
        epithelial.obsm['X_pca'].astype('float64'),
        epithelial.obs,
        'study',
        verbose=True,
    )
    epithelial.obsm['X_pca_harmony'] = ho.Z_corr
    print(f'  done ({time.time()-t0:.1f}s)')
    print(f'  embedding shape: {epithelial.obsm["X_pca_harmony"].shape}')

    # neighborhood graph
    print('Step 4/5 — neighborhood graph...')
    t0 = time.time()
    sc.pp.neighbors(epithelial, use_rep='X_pca_harmony', n_neighbors=10, n_pcs=20)
    print(f'  done ({time.time()-t0:.1f}s)')

    # UMAP and Leiden clustering
    print('Step 5/5 — UMAP and Leiden clustering...')
    t0 = time.time()
    sc.tl.umap(epithelial)
    sc.tl.leiden(epithelial, resolution=0.5)
    print(f'  done ({time.time()-t0:.1f}s)')

    # numeric sample ID for batch visualization
    epithelial.obs['sample_id'] = (
        epithelial.obs['sample']
        .map({s: i for i, s in enumerate(epithelial.obs['sample'].unique())})
        .astype(float)
    )

    # save epithelial object — subsequent sessions load this directly
    out_path = repo_root / cfg['outputs']['processed'] / 'luad_epithelial_harmony.h5ad'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    epithelial.write(out_path)
    print(f'Saved epithelial object → {out_path}')

    # save full LUAD object — all cell types, used for supplemental analyses
    # no Harmony needed here — used for expression comparisons, not UMAP
    luad = adata[
        adata.obs['disease'] == 'lung adenocarcinoma'
    ].copy()
    luad = luad[luad.obs['origin'].isin(['tumor_primary', 'normal_adjacent'])].copy()
    luad_path = repo_root / cfg['outputs']['processed'] / 'luad.h5ad'
    luad.write(luad_path)
    print(f'Saved full LUAD object → {luad_path}')

else:
    print('Harmony embedding already present — skipping reintegration')

## Analysis subsets

All downstream figure subsets are derived from `epithelial` here in one place,
so that UMAPs and expression comparisons always operate on the same cells.

In [ ]:
# paired LUAD donors — used for figure 2a (tumor vs NAT comparison)
luad = epithelial[epithelial.obs['origin'].isin(['tumor_primary', 'normal_adjacent'])].copy()

paired_donors = (
    luad.obs
    .groupby('donor_id')['origin']
    .nunique()
)
paired_donors = paired_donors[paired_donors == 2].index
luad_paired   = luad[luad.obs['donor_id'].isin(paired_donors)].copy()

print(f'Paired donors:   {len(paired_donors)}')
print(f'Cells retained:  {luad_paired.n_obs:,}')

In [ ]:
# cancer cells from primary tumor + metastasis — used for figure 2c
met_nonmet = epithelial[
    epithelial.obs['origin'].isin(['tumor_primary', 'tumor_metastasis']) &
    (epithelial.obs['ann_fine'] == 'Cancer cells')
].copy()

# primary tumor cancer cells only — used for figure 2d (stage comparison)
# stage mapped to early (I/II) vs late (III/IV)
nonmet = epithelial[
    (epithelial.obs['origin'] == 'tumor_primary') &
    (epithelial.obs['ann_fine'] == 'Cancer cells')
].copy()

nonmet.obs['stage_group'] = nonmet.obs['uicc_stage'].map(
    lambda x: 'Early' if x in ['I', 'II'] else 'Late'
)

print(f'met_nonmet:  {met_nonmet.n_obs:,} cells')
print(f'nonmet:      {nonmet.n_obs:,} cells')
print(f'Stage distribution:\n{nonmet.obs["stage_group"].value_counts()}')

## UMAPs

QC plots confirm Harmony integration quality before generating figure panels.
Figure UMAPs are saved to `outputs/figures/figure2/` and sit alongside the
corresponding expression comparison panels in the final figure layout.

In [ ]:
# QC — verify Harmony integration
# cells should mix across studies and samples with no strong batch clustering
sc.pl.umap(epithelial, color='sample_id', cmap='rainbow',
           title='Samples (batch check)', s=5, alpha=0.9)
sc.pl.umap(epithelial, color='study', title='Study (batch check)', s=5, alpha=0.5)

In [ ]:
# set scanpy figure output directory
sc.settings.figdir = str(fig_out)

In [ ]:
# figure 2 context UMAP — cancer vs non-cancer epithelial cells
sc.pl.umap(
    epithelial,
    color='cancer_noncancer',
    palette={'Cancer cells': 'tab:red', 'Epithelial cells': 'tab:grey'},
    title='Epithelial lineage',
    s=5, alpha=0.8,
    save='_figure2_umap_cancer_noncancer.pdf',
)


In [ ]:
# cancer cells from primary tumor + metastasis — used for figure 2c
met_nonmet = epithelial[
    epithelial.obs['origin'].isin(['tumor_primary', 'tumor_metastasis']) &
    (epithelial.obs['ann_fine'] == 'Cancer cells')
].copy()

# rename origin labels for display
met_nonmet.obs['origin'] = (
    met_nonmet.obs['origin']
    .astype('category')
    .cat.rename_categories({
        'tumor_primary':    'Primary Tumor',
        'tumor_metastasis': 'Metastasis',
    })
)

print(f'met_nonmet: {met_nonmet.n_obs:,} cells')
print(met_nonmet.obs['origin'].value_counts())


In [ ]:
# figure 2c context UMAP
sc.pl.umap(
    met_nonmet,
    color='origin',
    palette={'Primary Tumor': 'goldenrod', 'Metastasis': 'red'},
    title='Primary vs metastasis',
    s=3, alpha=0.7,
    show=False,
)

In [ ]:

# figure 2d context UMAP — early vs late stage primary tumor cancer cells
sc.pl.umap(
    nonmet,
    color='stage_group',
    palette = {'Early': '#f8a5c2', 'Late': '#34495e'},  # pink and deep navy blue
    title='Stage',
    s=10, alpha=0.5,
    save='_figure2d_umap_stage.pdf',
)



In [ ]:
# HLA-DRA expression on epithelial lineage
sc.pl.umap(
    epithelial,
    color='HLA-DRA',
    gene_symbols='feature_name',
    title='HLA-DRA',
    use_raw=False,
    cmap='inferno',
    s=2, alpha=0.8,
    save='_figure2_umap_hladra.pdf',
)

## Figure 2a — MHC II expression in malignant vs normal adjacent epithelial cells

Paired comparison of MHC II gene expression between primary tumor cancer cells
and matched normal adjacent epithelial cells. Only donors with both origins
represented are included. Wilcoxon signed-rank test on donor-level means.

In [ ]:
# figure 2a — core MHC II gene set
# nonparametric test used on all genes for consistency (distributions are mixed)
plot_genes_paired_luad(
    luad_paired,
    genes=mhc2_genes,
    test_mode='nonparametric',
    figsize_per_gene=(5, 5),
    save_path=fig_out / 'figure2a_normal_tumor.pdf',
)

In [ ]:
# figure 2a extended — accessory and co-stimulatory molecules
# shows that MHC II processing machinery is also expressed in malignant cells
plot_genes_paired_luad(
    luad_paired,
    genes=extended_genes,
    test_mode='nonparametric',
    nrows=2,
    figsize_per_gene=(5, 5),
    save_path=fig_out / 'figure2a_normal_tumor_extended.pdf',
)

## Figure 2b — MHC II accessory and co-stimulatory molecule expression in malignant cells

Compares expression of MHC II processing and co-stimulatory molecules between
primary tumor cancer cells and normal adjacent epithelial cells. Builds on
prior literature showing these molecules are required for AT2 cell antigen
presentation — their retention in malignant cells suggests preserved antigen
presentation capacity in a subset of tumors.

In [ ]:
# figure 2b — accessory molecules: ICAM1, HLA-DMA, HLA-DMB, CD80, CD86
# subset of extended_genes excluding core MHC II chains
accessory_genes = ['ICAM1', 'HLA-DMA', 'HLA-DMB','CD80', 'CD86']

plot_genes_paired_luad(
    luad_paired,
    genes=accessory_genes,
    test_mode='nonparametric',
    figsize_per_gene=(5, 5),
    save_path=fig_out / 'figure2b_accessory_molecules.pdf',
)

## Figure 2c — MHC II expression in primary tumor vs metastatic cancer cells

Compares sample-level mean MHC II expression between primary tumor and metastatic
cancer cells. Only donors with cells in both origins are included. Mann-Whitney U
test on sample-level means. A UMAP showing the cells being compared is saved
alongside the expression panels.

In [ ]:
# figure 2c UMAP — shows which cells are being compared
sc.pl.umap(
    met_nonmet,
    color='origin',
    palette={'Primary Tumor': 'goldenrod', 'Metastasis': 'red'},
    title='Primary vs metastasis',
    s=3, alpha=0.7,
    show=False,
)
plt.savefig(fig_out / 'figure2c_umap_met_vs_primary.pdf', bbox_inches='tight')
plt.savefig(fig_out / 'figure2c_umap_met_vs_primary.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# figure 2c — MHC II expression primary tumor vs metastasis
plot_scrna_group_comparison(
    met_nonmet,
    genes=ens_mhc2,
    group_col='origin',
    order=['Primary Tumor', 'Metastasis'],
    palette={'Primary Tumor': 'goldenrod', 'Metastasis': 'red'},
    xtick_labels=['Primary', 'Metastasis'],
    figsize_per_gene=(5, 5),
    nrows=1,
    save_path=fig_out / 'figure2c_met_vs_primary.pdf',
)

## Figure 2d — MHC II expression by disease stage in primary tumor cancer cells

Compares sample-level mean MHC II expression between early (stage I/II) and
late (stage III/IV) primary tumor cancer cells. Mann-Whitney U test on
sample-level means. UICC stage is mapped to early/late before aggregation.

In [ ]:
# figure 2d UMAP — shows which cells are being compared
sc.pl.umap(
    nonmet,
    color='stage_group',
    palette={'Early': '#f8a5c2', 'Late': '#34495e'},
    title='Stage',
    s=3, alpha=0.7,
    show=False,
)
plt.savefig(fig_out / 'figure2d_umap_stage.pdf', bbox_inches='tight')
plt.savefig(fig_out / 'figure2d_umap_stage.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# figure 2d — MHC II expression early vs late stage primary tumor
plot_scrna_group_comparison(
    nonmet,
    genes=ens_mhc2,
    group_col='stage_group',
    order=['Early', 'Late'],
    palette={'Early': '#f8a5c2', 'Late': '#34495e'},
    xtick_labels=['Early', 'Late'],
    figsize_per_gene=(5, 5),
    nrows=1,
    save_path=fig_out / 'figure2d_early_vs_late.pdf',
)